# Energy flexibility: the model's baseline, its emissions, and the two programme results

**Claim cluster:** `C051`, `C052`, `C053`, `C056`, `C059`, `C060`, `C090`, `C117`, `C118`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in these notebooks reads only from data/smarthome.duckdb.
CLAIM_IDS = ('C051', 'C052', 'C053', 'C056', 'C059', 'C060', 'C090', 'C117', 'C118')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

This cluster mixes modelled inputs with measured programme outcomes, and the extraction log records the model's baseline, sensitivity and payback values as scenario assumptions rather than measurements (decision `L034`). The panels below keep the two apart: everything in the model's baseline section is an input, and only the demand-response results are field measurements.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### What the flexibility model assumes a household consumes

The model's baseline section records annual consumption for five loads on one unit (kWh per year), which is the only part of the model that can share an axis. `M107` states a daily figure for the same household (30 kWh/day) and `M113`/`M114` state generation and storage capacity rather than consumption; all three are shown in the table and kept off the chart, and no total is computed, because the corpus records none and the daily and annual figures are different bases.

In [ ]:
# Only rows whose unit says kWh/year belong on this axis. M107 (kWh/day),
# M113 (kW; kWh/year generation) and M114 (kWh; kW capacity) are different
# quantities and stay in the table below.
BASELINE_SECTION = "1. System Boundary and Baseline"
baseline_rows = viz.section_metrics(BASELINE_SECTION)
show(baseline_rows[["local_id", "metric_name", "value", "unit", "confidence",
                    "conditions_stated"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "conditions_stated": "cond_stated"}))

CONSUMPTION_IDS = ["M108", "M109", "M110", "M111", "M112"]
consumption = viz.metrics_by_id(CONSUMPTION_IDS)
consumption["kwh_year"] = [viz.numeric_value(value)
                           for value in consumption["value"]]
consumption = consumption.dropna(subset=["kwh_year"])
show(consumption[["local_id", "metric_name", "value", "unit", "kwh_year",
                  "confidence"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "confidence": "conf"}))

fig = go.Figure(go.Bar(
    x=consumption["local_id"] + " " + consumption["metric_name"],
    y=consumption["kwh_year"],
    marker_color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                  for c in consumption["confidence"]],
    text=["%g" % value for value in consumption["kwh_year"]],
    textposition="outside",
    customdata=list(zip(consumption["value"], consumption["unit"],
                        consumption["confidence"])),
    hovertemplate=("%{x}<br>stated: %{customdata[0]} %{customdata[1]}"
                   "<br>confidence: %{customdata[2]}<extra></extra>"),
))
fig.update_layout(
    title=("Modelled baseline annual consumption by load (kWh/year). Model "
           "inputs, not measurements; no total is computed"),
    yaxis_title="kWh per year, as stated",
    height=380,
    xaxis=dict(tickangle=-20),
)
fig.show()

## Supporting context

### Emissions are modelled; the two programme results are not

`M127` records the emission factors the model applies, all in kg CO2 per kWh, so the three factors can share an axis; the reductions that follow from them (`M128`-`M130`) are modelled outputs and are shown as text. Against those stand the two field results in this cluster: the UK demand-flexibility programme reduced peak demand by `23.1%` among compliers (`M089`, recorded in wave 1's energy notebook) and by `28.1%` among all participants (`M090`). They are shown together because the gap between them is the participation effect, and the corpus states both populations explicitly.

In [ ]:
# Emission factors share one unit (kg CO2 per kWh), so they can be charted.
# The three labels are the three readings the value cell writes, in order.
_factors = viz.metrics_by_id(["M127"])
_numbers = viz.parse_numbers(_factors["value"].iloc[0]) if not _factors.empty else []
_factor_labels = ["average", "peak marginal", "off-peak marginal"][:len(_numbers)]
_factor_rows = pd.DataFrame({
    "reading": _factor_labels,
    "kg_co2_per_kwh": _numbers,
    "metric": _factors["local_id"].iloc[0] if not _factors.empty else "",
    "confidence": _factors["confidence"].iloc[0] if not _factors.empty else None,
})
show(_factor_rows)

fig = go.Figure(go.Bar(
    x=_factor_rows["reading"],
    y=_factor_rows["kg_co2_per_kwh"],
    marker_color=["#1f77b4", "#b06000", "#1a7f37"][:len(_factor_rows)],
    text=["%g" % value for value in _factor_rows["kg_co2_per_kwh"]],
    textposition="outside",
    customdata=list(zip(_factor_rows["metric"], _factor_rows["confidence"])),
    hovertemplate=("%{x}<br>%{y} kg CO2 per kWh (metric %{customdata[0]},"
                   " confidence %{customdata[1]})<extra></extra>"),
))
fig.update_layout(
    title=("Emission factors applied by the model (M127): one unit, three "
           "stated marginalities"),
    yaxis_title="kg CO2 per kWh",
    height=320,
)
fig.show()

# The modelled reductions are stated in different units (kg per year, tonnes
# per year, percent per household), so they are a table, not a chart.
display(HTML("<h4>Modelled emissions reductions</h4>"
             "<p>Modelled outputs: they follow from the factors above and the "
             "baseline consumption, not from measurement.</p>"))
show(viz.metrics_by_id(["M128", "M129", "M130"])[
    ["local_id", "metric_name", "value", "unit", "confidence", "source_ref"]]
     .rename(columns={"local_id": "metric", "metric_name": "name",
                      "source_ref": "source", "confidence": "conf"}))

display(HTML("<h4>Peak-demand reduction: compliers against all participants</h4>"
             "<p>Both figures come from the same nationwide UK field "
             "experiment; the populations differ, and the corpus states "
             "both.</p>"))
_peak = viz.metrics_by_id(["M089", "M090"])
_peak["population"] = ["compliers", "all participants"]
show(_peak[["local_id", "population", "metric_name", "value", "unit",
            "confidence", "source_ref"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))

fig = go.Figure(go.Bar(
    x=_peak["population"],
    y=[viz.numeric_value(value) for value in _peak["value"]],
    marker_color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                  for c in _peak["confidence"]],
    text=["%g%%" % viz.numeric_value(value) for value in _peak["value"]],
    textposition="outside",
    customdata=list(zip(_peak["local_id"], _peak["value"], _peak["source_ref"])),
    hovertemplate=("%{x}<br>stated: %{customdata[1]}% "
                   "(metric %{customdata[0]}, source %{customdata[2]})"
                   "<extra></extra>"),
))
fig.update_layout(
    title=("Measured peak-demand reduction in the UK flexibility programme, "
           "by stated population (M089, M090)"),
    yaxis_title="percent reduction",
    height=320,
)
fig.show()

display(HTML("<h4>The rest of the programme economics, as recorded</h4>"))
show(viz.metrics_by_id(["M091", "M092", "M093", "M153", "M125", "M126"])[
    ["local_id", "metric_name", "value", "unit", "confidence", "source_ref",
     "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))
display(HTML(
    "<h4>Claim cluster evidence table</h4>"
    "<p>Every claim in this cluster, with its evidence class, confidence, "
    "workflow stage and cited source ids. This is the table the figures above "
    "are built from.</p>"
))
show(
    claims[["local_id", "claim_type", "evidence_marker", "confidence",
            "workflow_stage", "source_refs", "falsifier_missing", "section"]]
    .rename(columns={
        "local_id": "claim", "claim_type": "type", "evidence_marker": "ev",
        "confidence": "conf", "workflow_stage": "stage",
        "source_refs": "sources", "falsifier_missing": "no_falsifier",
    })
)

display(HTML("<h4>Sources cited by the cluster</h4>"))
show(sources[["local_id", "title", "publisher", "classification",
              "publication_date", "url"]].rename(
    columns={"local_id": "source"}))

## Uncertainty and gaps

### Which numbers on this page are measured, and which are inputs

Only the two peak-demand figures come from a field experiment. The baseline consumption rows, the emission factors and the modelled reductions are model inputs or outputs, recorded at medium confidence because the extraction log classifies the model as scenario assumptions rather than measurements.

Other limits on this page:

- **Two consumption bases for one household.** `M107` states 30 kWh/day while `M108`-`M112` state annual figures for individual loads. The corpus never reconciles them, so this page does not either: no total is computed and the daily figure stays a table row.
- **No universal savings claim is available.** `C090` (a documented fact, recorded under interpretation warnings) states that the research does not prove universal energy savings: results hold under particular tariffs, climates, equipment and participation conditions.
- **`C056` is an inference, not a measurement.** The claim that demand-response compensation dominates the economics rather than energy arbitrage alone is the corpus's own reading of its scenario model.
- **The fleet emissions figure is not a forecast.** `M129` states 792,000 tonnes CO2/year by scaling a per-household figure, so it inherits every assumption in the model.
- **Solar and battery rows mix two units in one cell.** `M113` and `M114` state capacity and output together (kW and kWh), which is why they stay in the table rather than on an axis.

In [ ]:
# Model inputs are separated from field results by section, so a reader can see
# how much of the page rests on scenario assumptions rather than measurement.
display(HTML(
    "<h4>Where each modelled quantity sits in the corpus</h4>"
    "<p>The model spans four sections; the field results sit outside them.</p>"
))
_MODELLED_SECTIONS = ["1. System Boundary and Baseline",
                      "4. Core Financial Model", "7. Stress Tests",
                      "Payback and NPV"]
_modelled = pd.concat([viz.section_metrics(section)
                       for section in _MODELLED_SECTIONS], ignore_index=True)
show(_modelled.groupby("section").agg(
    metrics=("local_id", "count"),
    medium_confidence=("confidence", lambda s: int((s == "medium").sum())),
).reset_index().rename(columns={"section": "modelled_section"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Section-scoped rows come from `viz.section_metrics`; named rows from `viz.metrics_by_id`. Numbers are parsed with `viz.numeric_value` and `viz.parse_numbers`, and the three emission-factor labels are the three readings the value cell writes, in cell order.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()